# 📈 RELATÓRIO DOCHMO — ANÁLISE DE SÉRIES TEMPORAIS (v2.0)
### **Pipeline de Geração de Visualizações Interativas em HTML**

Este notebook consolida as rotinas de engenharia de visualização para o caso de estudo evoluído (v2.0) da produção mensal de petróleo no Brasil. A versão 2.0 incorpora melhorias metodológicas significativas em relação à v1.0, abrangendo a extensão dos dados de série histórica até o final de 2025 (totalizando 348 observações), o ajuste e correção por efeito de calendário, a marcação explícita de marcos regulatórios e choques macroeconômicos (intervenções), a análise de calibração empírica dos intervalos de confiança e previsões futuras para 2026.

**Índice de Visualizações:**
1. **Gráfico 01:** Série Histórica Completa da ANP (1997–2025) com área sob a curva dourada.
2. **Gráfico 02:** Desvios Sazonais Médios por Dia (Bruto vs. Ajustado por efeito calendário).
3. **Gráfico 03:** Linha do tempo de Intervenções e Choques de Apagão, Greves e Pandemia.
4. **Gráfico 04:** Competição Preditiva no Período de Teste/Holdout (SARIMA, ARIMAX, Fourier, Prophet, Holt-Winters).
5. **Gráfico 05:** Projeção de Cenários e Intervalos de Confiança (80% e 95%) para o ano de 2026 pelo Holt-Winters.

In [ ]:
# Importação de bibliotecas padrão de ciência de dados e plotagem interativa
import os
import pandas as pd
import plotly.graph_objects as go

# Importações do módulo central de temas corporativos DOCHMO (case_theme.py)
from case_theme import (
    apply_dochmo_theme, write_case_html,
    COLOR_GOLD, COLOR_GOLD_FAINT, COLOR_WHITE, COLOR_WHITE_SOFT,
    COLOR_WHITE_FAINT, COLOR_GREEN, COLOR_BLUE, COLOR_RED
)

## 📂 Configurações de Diretórios de Entrada/Saída

Estabelece os subdiretórios locais para leitura das tabelas pré-computadas (`/case`) e gravação das visualizações exportadas em HTML (`/graphics`).

In [ ]:
# Nome dos diretórios de dados locais e saída física dos gráficos
DIR_CASE = "../datasets"
DIR_OUT = "../graphics"

# Garante a criação física da pasta de saída caso não exista no deploy
os.makedirs(DIR_OUT, exist_ok=True)

In [ ]:
def path_case(nome):
    """Retorna o caminho resolvido para um dataset de entrada na pasta /case"""
    return os.path.join(DIR_CASE, nome)
 
def path_out(nome):
    """Retorna o caminho de escrita física para os arquivos HTML de saída na pasta /graphics"""
    return os.path.join(DIR_OUT, nome)

## 📊 01. Série Histórica Completa da ANP (1997–2025)

Esta visualização apresenta o volume total bruto de petróleo produzido mensalmente no Brasil. A curva expõe a decolagem da produção brasileira impulsionada pelo Pré-Sal no início da década de 2010. Utiliza uma área sombreada dourada suave sob a curva para conferir volume e elegância.

In [ ]:
def grafico_01_serie_historica():
    # Carrega a série histórica completa de produção em m³
    df = pd.read_csv(path_case("case_serie_completa.csv"), parse_dates=["data"])
 
    # Tradução e parsing dos meses para compor rótulos legíveis em português no hover
    months_pt = [
        "Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
        "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro"
    ]
    hover_dates = [f"{months_pt[dt.month - 1]} de {dt.year}" for dt in df["data"]]

    fig = go.Figure()
    # Linha principal com preenchimento até a linha zero (estilo área sombreada)
    fig.add_trace(go.Scatter(
        x=df["data"], y=df["producao_m3"],
        mode="lines",
        line=dict(color=COLOR_GOLD, width=2),
        fill="tozeroy",
        fillcolor=COLOR_GOLD_FAINT, # Dourado institucional muito suave de fundo
        name="Produção mensal",
        customdata=hover_dates,
        hovertemplate="<b>%{customdata}</b><br>Volume: %{y:,.0f} m³<extra></extra>",
    ))
    
    # Aplica o tema visual unificado da DOCHMO
    apply_dochmo_theme(fig, height=460)
    
    # Sobrescreve parâmetros específicos para este gráfico (Título, Tipografia e Labels dos Eixos)
    fig.update_layout(
        title=dict(
            text="Série Histórica da Produção Mensal de Petróleo (1997–2025)",
            font=dict(size=20, family="Playfair Display", color=COLOR_GOLD),
            x=0.5
        ),
        font=dict(family="Inter, Arial", size=14, color="#F8F9FA"),
        xaxis_title="Tempo",
        yaxis_title="Volume Produzido (m³)",
        separators=",."
    )
    
    # Gridlines douradas discretas para facilitar a leitura macro
    fig.update_xaxes(showgrid=True, gridcolor="rgba(215, 181, 109, 0.1)")
    fig.update_yaxes(
        showgrid=True, gridcolor="rgba(215, 181, 109, 0.1)",
        tickfont=dict(color="#F8F9FA"),
        tickformat=",.0f"
    )
 
    # Exportação final standalone em HTML
    write_case_html(fig, path_out("g01_serie_historica.html"))

In [ ]:
# Executa a função geradora do Gráfico 01
grafico_01_serie_historica()

## 📅 02. Efeito Calendário: Desvio Bruto vs. Ajustado por Dia

O efeito de calendário ocorre porque meses diferentes possuem números de dias diferentes (ex: Fevereiro com 28 e Dezembro com 31). Um mês de 31 dias naturalmente registrará uma produção maior do que um mês mais curto, mesmo que o ritmo diário seja o mesmo. 
Para expor esse efeito, comparamos o desvio percentual sazonal em relação à média geral utilizando:
*   **Desvio Bruto:** Volume sazonal bruto sem tratamento.
*   **Ajustado por dia:** Volume dividido pelo número real de dias do respectivo mês, isolando e purificando a sazonalidade.

In [ ]:
def grafico_02_efeito_calendario():
    # Carrega a tabela com os cálculos de desvio sazonal em percentual
    df = pd.read_csv(path_case("case_efeito_calendario.csv"))
    df["mes"] = pd.Categorical(df["mes"], categories=df["mes"], ordered=True)

    months_pt = [
        "Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
        "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro"
    ]

    fig = go.Figure()
    # Trace do Desvio Bruto (Branco translúcido de fundo para atenuar)
    fig.add_trace(go.Bar(
        x=df["mes"], y=df["desvio_pct_bruto"],
        name="Desvio bruto",
        marker_color=COLOR_WHITE_FAINT, marker_line_width=0,
        customdata=months_pt,
        hovertemplate="<b>%{customdata}</b><br>Bruto: %{y:.2f}%<extra></extra>",
    ))
    # Trace do Desvio Ajustado por Dia (Dourado de destaque na frente)
    fig.add_trace(go.Bar(
        x=df["mes"], y=df["desvio_pct_ajustado"],
        name="Ajustado por dia",
        marker_color=COLOR_GOLD, marker_line_width=0,
        customdata=months_pt,
        hovertemplate="<b>%{customdata}</b><br>Ajustado: %{y:.2f}%<extra></extra>",
    ))
    # Adiciona linha horizontal de referência na média zero
    fig.add_hline(y=0, line_width=1, line_color="rgba(248,249,250,0.3)")

    # Aplica o tema visual unificado da DOCHMO
    apply_dochmo_theme(fig, height=440)

    # Layout em modo agrupado ('group') para exibir as barras lado a lado e margem superior
    fig.update_layout(
        barmode="group",
        title=dict(
            text="Efeito Calendário: Desvio Bruto vs. Ajustado por Dia",
            font=dict(size=20, family="Playfair Display", color=COLOR_GOLD),
            x=0.5,
            y=0.95
        ),
        margin=dict(t=90), # Afasta o título superior para evitar sobreposição
        font=dict(family="Inter, Arial", size=14, color="#F8F9FA"),
        xaxis_title="Mês",
        yaxis_title="Desvio frente à média geral (%)",
        separators=",."
    )
    fig.update_yaxes(
        gridcolor="rgba(215, 181, 109, 0.1)",
        ticksuffix="%"
    )
 
    # Exportação em HTML
    write_case_html(fig, path_out("g02_efeito_calendario.html"))

In [ ]:
# Executa a função geradora do Gráfico 02
grafico_02_efeito_calendario()

## 🛠️ 03. Linha do Tempo de Intervenções e Choques na Produção

Este gráfico apresenta a série temporal histórica mapeando de forma explícita datas críticas de quebras estruturais e choques macroeconômicos identificados analiticamente (como o apagão do setor elétrico em 2001, o início oficial da produção comercial no Pré-Sal em 2010, greves de caminhoneiros e os impactos industriais da pandemia de COVID-19). Esses choques foram corrigidos na modelagem final utilizando variáveis regressores de intervenção (*dummies*).

In [ ]:
def grafico_03_intervencoes():
    # Carrega a série completa e a tabela de metadados das intervenções estimadas
    df = pd.read_csv(path_case("case_serie_completa.csv"), parse_dates=["data"])
    eventos = pd.read_csv(path_case("case_intervencoes.csv"), parse_dates=["data"])
 
    months_pt = [
        "Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
        "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro"
    ]
    hover_dates = [f"{months_pt[dt.month - 1]} de {dt.year}" for dt in df["data"]]

    fig = go.Figure()
    # Série histórica base plotada em linha branca suave semi-transparente
    fig.add_trace(go.Scatter(
        x=df["data"], y=df["producao_m3"],
        mode="lines",
        line=dict(color=COLOR_WHITE_SOFT, width=1.6),
        customdata=hover_dates,
        hovertemplate="<b>%{customdata}</b><br>Volume: %{y:,.0f} m³<extra></extra>",
        showlegend=False,
    ))
 
    # Alturas específicas em fator decimal (y_max * factor) planejadas para impedir
    # sobreposições visuais entre caixas de anotação muito próximas no tempo
    event_heights = [0.95, 0.85, 0.95, 0.75, 0.88]
    
    y_max = df["producao_m3"].max()
    # Itera sobre os eventos históricos desenhando as linhas verticais e anotações
    for i, row in eventos.iterrows():
        h_factor = event_heights[i] if i < len(event_heights) else (0.97 - 0.09 * (i % 2))
        
        # Trata e corrige caracteres acentuados que costumam quebrar em leituras cruas de CSV
        label_text = row["label"]
        label_text = label_text.replace("Apago", "Apagão").replace("Incio", "Início").replace("Pr-Sal", "Pré-Sal").replace("Regulatrio", "Regulatório")
        
        # Adiciona a linha vertical pontilhada de intervenção
        fig.add_vline(x=row["data"], line_width=1.4, line_dash="dash",
                      line_color=COLOR_GOLD, opacity=0.6)
        
        # Insere a caixa de texto dourada contendo o nome da ocorrência histórica
        fig.add_annotation(
            x=row["data"], y=y_max * h_factor,
            text=label_text,
            showarrow=False,
            font=dict(color=COLOR_GOLD, size=11, family="Inter, sans-serif"),
            bgcolor="rgba(10,20,40,0.85)", # Fundo azul muito escuro para contraste do texto
            bordercolor=COLOR_GOLD, borderwidth=1, borderpad=4,
            xanchor="left",
        )
 
    # Aplica o tema visual unificado da DOCHMO
    apply_dochmo_theme(fig, height=480)
    
    fig.update_layout(
        title=dict(
            text="Eventos de Intervenção na Produção de Petróleo",
            font=dict(size=20, family="Playfair Display", color=COLOR_GOLD),
            x=0.5
        ),
        font=dict(family="Inter, Arial", size=14, color="#F8F9FA"),
        xaxis_title="Tempo",
        yaxis_title="Volume Produzido (m³)",
        separators=",."
    )
    fig.update_xaxes(showgrid=True, gridcolor="rgba(215, 181, 109, 0.1)")
    fig.update_yaxes(
        showgrid=True, gridcolor="rgba(215, 181, 109, 0.1)",
        tickfont=dict(color="#F8F9FA"),
        tickformat=",.0f"
    )
 
    # Exportação em HTML
    write_case_html(fig, path_out("g03_intervencoes.html"))

In [ ]:
# Executa a função geradora do Gráfico 03
grafico_03_intervencoes()

## 🏆 04. Competição de Modelos Preditivos no Período de Holdout (2023–2025)

Esta visualização compara o desempenho das previsões fora da amostra (*out-of-sample*) geradas por diferentes algoritmos estatísticos (SARIMA clássico, ARIMAX com dummies, ARIMA estrutural com componentes de Fourier, algoritmo Prophet da Meta e o campeão empírico Holt-Winters) confrontando-os com os **Dados Reais Observados** (curva branca contínua) durante o horizonte de holdout de 36 meses.

In [ ]:
def grafico_04_comparacao_modelos():
    # Carrega a série histórica e as previsões computadas no período de holdout
    serie = pd.read_csv(path_case("case_serie_completa.csv"), parse_dates=["data"])
    holdout = pd.read_csv(path_case("case_holdout_previsoes.csv"), parse_dates=["data"])

    data_inicio_teste = holdout["data"].min()
    # Filtra os últimos 24 meses do treino histórico para dar contexto visual de transição
    historico_recente = serie[serie["data"] < data_inicio_teste].tail(24)

    months_pt = [
        "Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
        "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro"
    ]
    hover_dates_holdout = [f"{months_pt[dt.month - 1]} de {dt.year}" for dt in holdout["data"]]
    hover_dates_hist = [f"{months_pt[dt.month - 1]} de {dt.year}" for dt in historico_recente["data"]]

    fig = go.Figure()

    # 1. Área sombreada do Intervalo de Confiança a 95% do Holt-Winters (verde translúcido muito suave)
    fig.add_trace(go.Scatter(
        x=pd.concat([holdout["data"], holdout["data"].iloc[::-1]]),
        y=pd.concat([holdout["hw_sup95"], holdout["hw_inf95"].iloc[::-1]]),
        fill="toself", fillcolor="rgba(76, 175, 80, 0.10)",
        line=dict(width=0), hoverinfo="skip", showlegend=False,
    ))
    # 2. Área sombreada do Intervalo de Confiança a 80% do Holt-Winters (verde com opacidade maior)
    fig.add_trace(go.Scatter(
        x=pd.concat([holdout["data"], holdout["data"].iloc[::-1]]),
        y=pd.concat([holdout["hw_sup80"], holdout["hw_inf80"].iloc[::-1]]),
        fill="toself", fillcolor="rgba(76, 175, 80, 0.18)",
        line=dict(width=0), hoverinfo="skip", showlegend=False,
    ))

    # 3. Série histórica de Treino (Linha tracejada branca de apoio)
    fig.add_trace(go.Scatter(
        x=historico_recente["data"], y=historico_recente["producao_m3"],
        mode="lines", line=dict(color=COLOR_WHITE_SOFT, width=1.4, dash="dot"),
        name="Histórico recente",
        customdata=hover_dates_hist,
        hovertemplate="<b>%{customdata}</b><br>Volume: %{y:,.0f} m³<extra></extra>",
    ))
 
    # 4. Dados reais observados no teste (Curva contínua branca de alto destaque)
    fig.add_trace(go.Scatter(
        x=holdout["data"], y=holdout["observado"],
        mode="lines", line=dict(color=COLOR_WHITE, width=2.4),
        name="Observado",
        customdata=hover_dates_holdout,
        hovertemplate="<b>%{customdata}</b><br>Volume: %{y:,.0f} m³<extra></extra>",
    )) 

    # 5. Modelo Campeão: Holt-Winters (Linha verde contínua em destaque)
    fig.add_trace(go.Scatter(
        x=holdout["data"], y=holdout["hw"],
        mode="lines", line=dict(color="#1E6B3C", width=2.5),
        name="Holt-Winters (campeão)",
        customdata=hover_dates_holdout,
        hovertemplate="<b>%{customdata}</b><br>Previsão HW: %{y:,.0f} m³<extra></extra>",
    ))

    # 6. Modelo auto.arima (Linha bege/ocre)
    fig.add_trace(go.Scatter(
        x=holdout["data"], y=holdout["auto_arima"],
        mode="lines", line=dict(color="#BF8E4A", width=1.95),
        name="auto.arima",
        customdata=hover_dates_holdout,
        hovertemplate="<b>%{customdata}</b><br>Previsão auto.arima: %{y:,.0f} m³<extra></extra>",
    ))

    # 7. Modelo ARIMA + Fourier (Linha vermelha contínua)
    fig.add_trace(go.Scatter(
        x=holdout["data"], y=holdout["fourier"],
        mode="lines", line=dict(color="#d62728", width=2.3),
        name="ARIMA + Fourier",
        customdata=hover_dates_holdout,
        hovertemplate="<b>%{customdata}</b><br>Previsão Fourier: %{y:,.0f} m³<extra></extra>",
    ))

    # 8. Modelo ARIMAX com 5 variáveis de intervenção (Linha azul marinho)
    fig.add_trace(go.Scatter(
        x=holdout["data"], y=holdout["arimax_5m"],
        mode="lines", line=dict(color="#153269", width=2.2),
        name="ARIMAX (5 regressores)",
        customdata=hover_dates_holdout,
        hovertemplate="<b>%{customdata}</b><br>Previsão ARIMAX: %{y:,.0f} m³<extra></extra>",
    ))

    # 9. Modelo SARIMA(8,1,1)(1,0,1)12 estatístico (Linha azul claro)
    fig.add_trace(go.Scatter(
        x=holdout["data"], y=holdout["sarima"],
        mode="lines", line=dict(color="#1f77b4", width=2.1),
        name="SARIMA(8,1,1)(1,0,1)₁₂",
        customdata=hover_dates_holdout,
        hovertemplate="<b>%{customdata}</b><br>Previsão SARIMA: %{y:,.0f} m³<extra></extra>",
    ))

    # 10. Algoritmo Prophet da Meta (Linha marrom bordô)
    fig.add_trace(go.Scatter(
        x=holdout["data"], y=holdout["prophet"],
        mode="lines", line=dict(color="#450c06", width=1.8),
        name="Prophet (Meta)",
        customdata=hover_dates_holdout,
        hovertemplate="<b>%{customdata}</b><br>Previsão Prophet: %{y:,.0f} m³<extra></extra>",
    ))

    # Linha vertical pontilhada dividindo a transição histórico vs. holdout experimental
    fig.add_vline(x=data_inicio_teste, line_width=1, line_dash="dot",
                    line_color="rgba(248,249,250,0.35)")

    # Aplica o tema visual unificado da DOCHMO
    apply_dochmo_theme(fig, height=520)

    # Sobrescreve configurações estéticas do layout e adiciona a linha de controle vertical (spike)
    fig.update_layout(
        hovermode="x unified", # Habilita painel tooltip único contendo todos os modelos ao mesmo tempo
        title=dict(
            text="Comparação de Modelos no Período de Teste (2023–2025)",
            font=dict(size=20, family="Playfair Display", color=COLOR_GOLD),
            x=0.5,
            y=0.97
        ),
        margin=dict(t=120), # Margem ampla para abrigar a legenda horizontal contendo 7 modelos
        font=dict(family="Inter, Arial", size=14, color="#F8F9FA"),
        xaxis_title="Tempo",
        yaxis_title="Volume Produzido (m³)",
        separators=",.",
        paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="#0A1428"
    )
    fig.update_xaxes(
        showgrid=True,
        gridcolor="rgba(215, 181, 109, 0.1)",
        showspikes=True, # Linha de auxílio que corre verticalmente sob a mira do cursor
        spikecolor="#D7B56D",
        spikethickness=1,
        spikedash="dash",
        spikemode="across"
    )
    fig.update_yaxes(
        showgrid=True,
        gridcolor="rgba(215, 181, 109, 0.1)",
        tickfont=dict(color="#F8F9FA"),
        tickformat=",.0f"
    )
 
    # Exportação em HTML
    write_case_html(fig, path_out("g04_comparacao_modelos.html"))

In [ ]:
# Executa a função geradora do Gráfico 04
grafico_04_comparacao_modelos()

## 🔮 05. Previsão da Produção de Petróleo para 2026

Esta visualização apresenta o cenário oficial de previsões futuras para os 12 meses de 2026. Utiliza o algoritmo campeão **Holt-Winters Aditivo** com a incerteza futura mensurada por intervalos de confiança (IC) a 80% (área sombreada verde interna) e 95% (área sombreada verde externa), refletindo a amplitude estatística correta calculada no estudo.

In [ ]:
def grafico_05_previsao_2026():
    # Carrega o histórico de produção e as projeções futuras de 2026 com intervalos de confiança
    serie = pd.read_csv(path_case("case_serie_completa.csv"), parse_dates=["data"])
    prev = pd.read_csv(path_case("case_2026_hw_ic.csv"), parse_dates=["data"])
 
    # Filtra o histórico a partir de 2023 para focar nos anos recentes de maior interesse
    historico_recente = serie[serie["data"] >= "2023-01-01"]
 
    months_pt = [
        "Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
        "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro"
    ]
    hover_dates_hist = [f"{months_pt[dt.month - 1]} de {dt.year}" for dt in historico_recente["data"]]
    hover_dates_prev = [f"{months_pt[dt.month - 1]} de {dt.year}" for dt in prev["data"]]

    fig = go.Figure()
    # 1. Área sombreada do Intervalo de Confiança a 95% (verde mais claro de fundo)
    fig.add_trace(go.Scatter(
        x=pd.concat([prev["data"], prev["data"].iloc[::-1]]),
        y=pd.concat([prev["sup95"], prev["inf95"].iloc[::-1]]),
        fill="toself", fillcolor="rgba(76, 175, 80, 0.15)",
        line=dict(width=0), hoverinfo="skip", name="IC 95%",
        legendrank=40
    ))
    # 2. Área sombreada do Intervalo de Confiança a 80% (verde escuro de suporte)
    fig.add_trace(go.Scatter(
        x=pd.concat([prev["data"], prev["data"].iloc[::-1]]),
        y=pd.concat([prev["sup80"], prev["inf80"].iloc[::-1]]),
        fill="toself", fillcolor="rgba(76, 175, 80, 0.30)",
        line=dict(width=0), hoverinfo="skip", name="IC 80%",
        legendrank=30
    ))
    # 3. Curva histórica observada recente (Linha branca contínua)
    fig.add_trace(go.Scatter(
        x=historico_recente["data"], y=historico_recente["producao_m3"],
        mode="lines", line=dict(color=COLOR_WHITE, width=2),
        name="Histórico (2023–2025)",
        customdata=hover_dates_hist,
        hovertemplate="<b>%{customdata}</b><br>Volume: %{y:,.0f} m³<extra></extra>",
        legendrank=10
    ))
    # 4. Projeção central de Previsão 2026 (Linha verde contínua com marcadores)
    fig.add_trace(go.Scatter(
        x=prev["data"], y=prev["previsao"],
        mode="lines+markers", line=dict(color=COLOR_GREEN, width=2.4),
        marker=dict(size=5),
        name="Previsão 2026 (Holt-Winters)",
        customdata=hover_dates_prev,
        hovertemplate="<b>%{customdata}</b><br>Previsão: %{y:,.0f} m³<extra></extra>",
        legendrank=20
    ))
 
    # Divisória vertical no início de 2026 ligando a previsão pontual
    fig.add_vline(x=pd.Timestamp("2026-01-01"), line_width=1, line_dash="dot",
                  line_color=COLOR_GOLD)
 
    # Aplica o tema visual unificado da DOCHMO
    apply_dochmo_theme(fig, height=460)
    
    fig.update_layout(
        title=dict(
            text="Previsão da Produção de Petróleo para 2026",
            font=dict(size=20, family="Playfair Display", color=COLOR_GOLD),
            x=0.5,
            y=0.95
        ),
        margin=dict(t=90), # Aumenta a margem superior para o título
        font=dict(family="Inter, Arial", size=14, color="#F8F9FA"),
        xaxis_title="Tempo",
        yaxis_title="Volume Produzido (m³)",
        separators=",."
    )
    fig.update_xaxes(showgrid=True, gridcolor="rgba(215, 181, 109, 0.1)")
    fig.update_yaxes(
        showgrid=True, gridcolor="rgba(215, 181, 109, 0.1)",
        tickfont=dict(color="#F8F9FA"),
        tickformat=",.0f"
    )
 
    # Exportação em HTML
    write_case_html(fig, path_out("g05_previsao_2026.html"))

In [ ]:
# Executa a função geradora do Gráfico 05
grafico_05_previsao_2026()

In [ ]:
print("\n[OK] Todos os 5 gráficos da v2.0 foram gerados e exportados na pasta ./graphics/")